# Imports

In [3]:
import pandas as pd
import numpy as np

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.tree import DecisionTreeRegressor

# Read the data

In [5]:
df = pd.read_csv("/kaggle/input/datasets/ziad23samer/cleaned-brainrot/cleaned.csv")

In [6]:
X = df.drop(columns=["brain_rot_index"]).values
y = df["brain_rot_index"].values

In [7]:
print(X.shape)

(468671, 29)


In [8]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

In [9]:
print(X_train.shape, X_val.shape, X_test.shape)

(281202, 29) (93734, 29) (93735, 29)


# Models

## Baseline Model

In [10]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)

LinearRegression()

In [11]:
y_pred = baseline.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print(f"MSE: {mse}")

MSE: 20.858028823045217


## Advanced Imports

In [12]:
from sklearn.model_selection import (
    RepeatedKFold, GridSearchCV, cross_val_score
)

In [13]:
rkf = RepeatedKFold(n_splits=5, n_repeats=3, random_state=42)

In [14]:
def get_cv_scores(model, X, y):
    scores = cross_val_score(
        model, X, y,
        cv=rkf, scoring="r2", n_jobs=-1
    )
    return scores

## Linear Regression

In [15]:
from sklearn.linear_model import Ridge

In [16]:
param_grid = {
    "alpha": [0.1, 1, 10]
}

In [17]:
ridge = Ridge()

In [18]:
scores = get_cv_scores(ridge, X_train, y_train)

In [19]:
print(scores)

[0.71027126 0.71057129 0.71142486 0.70954272 0.70583087 0.70941289
 0.70791921 0.70823753 0.71128022 0.71075997 0.70972083 0.70901957
 0.71118669 0.70974067 0.70796158]


In [20]:
X_trainval = np.vstack([X_train, X_val])
y_trainval = np.concatenate([y_train, y_val])

In [21]:
grid_search = GridSearchCV(
        estimator  = ridge,
        param_grid = param_grid,
        cv         = rkf,
        scoring    = "r2",
        n_jobs     = -1,
        verbose    = 0,
        refit      = True,
)
grid_search.fit(X_trainval, y_trainval)

best_LR = grid_search.best_estimator_
print({
    "best_params": grid_search.best_params_,
    "best_cv_r2" : grid_search.best_score_,
})

{'best_params': {'alpha': 10}, 'best_cv_r2': np.float64(0.7095124955820695)}


In [22]:
y_pred = best_LR.predict(X_test)

In [23]:
mse = mean_squared_error(y_pred, y_test)
print(f"Mean Squared Error of the linear regression is {mse}")

Mean Squared Error of the linear regression is 20.856728341553463


## AdaBoost Regressor

In [ ]:
from sklearn.ensemble import AdaBoostRegressor

In [ ]:
param_grid = {
    "n_estimators":  [50, 100, 200],
    "learning_rate": [0.01, 0.1, 0.5, 1.0],
    "loss":          ["linear", "square", "exponential"],
}

In [ ]:
ada = AdaBoostRegressor()

In [ ]:
scores = get_cv_scores(ada, X_train, y_train)

In [ ]:
print(scores)

In [ ]:
grid_search = GridSearchCV(
        estimator  = ada,
        param_grid = param_grid,
        cv         = 5,
        scoring    = "r2",
        n_jobs     = -1,
        verbose    = 0,
        refit      = True,
)
grid_search.fit(X_trainval, y_trainval)

best_ada = grid_search.best_estimator_
print({
    "best_params": grid_search.best_params_,
    "best_cv_r2" : grid_search.best_score_,
})

In [ ]:
y_pred = best_ada.predict(X_test)
mse = mean_squared_error(y_pred, y_test)
print(f"Mean Squared Error of the linear regression is {mse}")

## SVM

In [ ]:
from sklearn.svm import SVR

In [ ]:
from sklearn.linear_model import SGDRegressor
svm = SGDRegressor(loss='epsilon_insensitive')


In [ ]:
scores = get_cv_scores(svm, X_train, y_train)
print(scores)

In [ ]:
"""
SGDRegressor(loss='squared_error', *, penalty='l2',
alpha=0.0001, l1_ratio=0.15, fit_intercept=True,
max_iter=1000, tol=0.001, shuffle=True, verbose=0, epsilon=0.1,
random_state=None, learning_rate='invscaling', eta0=0.01, power_t=0.25,
early_stopping=False, validation_fraction=0.1, n_iter_no_change=5, warm_start=False, average=False)
"""

In [ ]:
param_grid = {
    "C":       [0.1, 1, 10, 100],
    "epsilon": [0.01, 0.1, 0.5],
    "kernel":  ["rbf", "linear"],
    "gamma":   ["scale", "auto"]
}

In [ ]:
grid_search = GridSearchCV(
        estimator  = svm,
        param_grid = param_grid,
        cv         = rkf,
        scoring    = "r2",
        n_jobs     = -1,
        verbose    = 0,
        refit      = True,
)
grid_search.fit(X_trainval, y_trainval)

best_SVM = grid_search.best_estimator_
print({
    "best_params": grid_search.best_params_,
    "best_cv_r2" : grid_search.best_score_,
})

## Bagging

### RT

In [24]:
from sklearn.ensemble import BaggingRegressor
from sklearn.tree import DecisionTreeRegressor


In [35]:
DT = DecisionTreeRegressor()
bagging_regressor = BaggingRegressor(DT, n_estimators=10, random_state=42)
bagging_regressor.fit(X_train, y_train)


BaggingRegressor(estimator=DecisionTreeRegressor(), random_state=42)

In [36]:
y_pred = bagging_regressor.predict(X_val)
mse = mean_squared_error(y_val, y_pred)
print("val MSE: ", mse)

val MSE:  24.188814991779584


In [37]:
y_pred = bagging_regressor.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print("test MSE: ", mse)

test MSE:  24.109317231114968


### LR

In [27]:
bagging_regressor = BaggingRegressor(best_LR, n_estimators=10, random_state=42)
bagging_regressor.fit(X_train, y_train)

BaggingRegressor(estimator=Ridge(alpha=10), random_state=42)

In [28]:
y_pred = bagging_regressor.predict(X_val)
mse = mean_squared_error(y_val, y_pred)
print("val MSE: ", mse)

val MSE:  20.998787182051867


In [29]:
from sklearn.model_selection import RandomizedSearchCV

### LR Bagging GridSearch

In [40]:
param_grid = {
    "n_estimators":       [5, 10, 20, 50],
    "max_samples": [0.2, 0.5, 0.7, 1.0],
    "max_features":  [0.2, 0.5, 0.7, 1.0],
    "bootstrap_features":   [True, False]
}
clf = RandomizedSearchCV(
    BaggingRegressor(Ridge(alpha=1)),
    param_grid, 
    n_iter=30,      
    scoring="neg_mean_squared_error", 
    cv=5,
    n_jobs=-1,
    verbose=3,
    pre_dispatch='2*n_jobs',
    random_state=42      
)
clf.fit(X_trainval, y_trainval)

best_bagging_estimator = clf.best_estimator_


Fitting 5 folds for each of 30 candidates, totalling 150 fits
{'best_params': {'n_estimators': 10, 'max_samples': 1.0, 'max_features': 1.0, 'bootstrap_features': False}, 'best_cv_r2': np.float64(21.028143661687825)}


In [43]:
print({
    "best_params": clf.best_params_,
    "best_cv_mse" : clf.best_score_ * -1,
})

{'best_params': {'n_estimators': 10, 'max_samples': 1.0, 'max_features': 1.0, 'bootstrap_features': False}, 'best_cv_mse': np.float64(21.028143661687825)}


In [42]:
y_pred = best_bagging_estimator.predict(X_test)
mse = mean_squared_error(y_pred, y_test)
print(f"Mean Squared Error of the linear regression bagging is {mse}")

Mean Squared Error of the linear regression bagging is 20.857263959043607


## ACHIEVeD MSE